# AAMI 良率智能助手 MVP Demo
## 电镀工序高风险批次识别与原因解释

## 第 1 部分：项目背景
- 为什么从良率切入：良率是最直接的经营与质量指标，且 lot 级数据具备可操作性。
- 为什么不先做完整 MES：本 Demo 强调最小可用闭环，避免范围失控。
- 本次只做一个闭环：**数据 -> 风险识别 -> 原因解释 -> 建议动作 -> 回写入口**。

## 第 2 部分：场景故事
- AAMI 深圳工厂电镀车间，正常良率通常在 **96%–98%**。
- 当天已有异常批次：**LOT1006、LOT1007**。
- **LOT1008** 尚未完成终检（PENDING），但 AI 需要提前预警。

In [ ]:
# 基础依赖
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

sys.path.append("../src")
from llm_provider import get_llm_provider, generate_lot_explanation, generate_action_advice

plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
# 数据文件准备：若不存在原始CSV，则自动创建
from pathlib import Path

raw_path = Path("../data/aami_yield_demo.csv")
clean_path = Path("../data/aami_yield_demo_clean.csv")
raw_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_path.exists():
    rows = [
        ["HIST","LOT0918","M1","E01","Day",1,118,62,45.1,"MB01",97.4,0,"none","DONE"],
        ["HIST","LOT0919","M1","E01","Night",2,121,61,45.4,"MB01",97.0,0,"none","DONE"],
        ["HIST","LOT0920","M2","E02","Day",1,119,63,44.9,"MB02",96.8,0,"none","DONE"],
        ["HIST","LOT0921","M2","E03","Night",3,134,66,46.2,"MB02",91.4,1,"plating_void","DONE"],
        ["HIST","LOT0922","M1","E02","Day",2,120,62,45.0,"MB03",97.1,0,"none","DONE"],
        ["HIST","LOT0923","M3","E01","Night",1,122,61,45.3,"MB04",96.6,0,"none","DONE"],
        ["HIST","LOT0924","M3","E02","Day",2,118,60,44.8,"MB04",97.6,0,"none","DONE"],
        ["HIST","LOT0925","M2","E03","Night",2,132,65,46.0,"MB02",92.2,1,"burn_mark","DONE"],
        ["HIST","LOT0934","M2","E03","Night",4,136,67,46.4,"MB02",90.8,1,"edge_peeling","DONE"],
        ["HIST","LOT0945","M2","E03","Night",3,133,66,46.1,"MB07",91.7,1,"plating_void","DONE"],
        ["CURRENT","LOT1006","M2","E03","Night",3,137,67,46.5,"MB10",89.9,1,"burn_mark","DONE"],
        ["CURRENT","LOT1007","M2","E03","Night",4,135,66,46.3,"MB10",90.6,1,"edge_peeling","DONE"],
        ["CURRENT","LOT1008","M2","E03","Night",3,136,66,46.2,"MB11",None,None,None,"PENDING"],
        ["CURRENT","LOT1009","M1","E01","Day",1,120,61,45.0,"MB12",None,None,None,"PENDING"],
        ["CURRENT","LOT1010","M3","E02","Day",2,122,62,45.1,"MB13",None,None,None,"PENDING"],
    ]
    cols = ["record_type","lot_id","prod_model","machine_id","shift","setup_batch_seq","plating_current_a",
            "plating_time_s","bath_temp_c","material_batch","yield_pct","defect_flag","defect_type","inspection_status"]
    pd.DataFrame(rows, columns=cols).to_csv(raw_path, index=False)

# 清洗并保存 clean 版本
raw_df = pd.read_csv(raw_path)
for c in ["record_type","lot_id","prod_model","machine_id","shift","material_batch","defect_type","inspection_status"]:
    raw_df[c] = raw_df[c].astype(str).str.strip().replace({"nan": None})

for c in ["setup_batch_seq","plating_current_a","plating_time_s","bath_temp_c","yield_pct","defect_flag"]:
    raw_df[c] = pd.to_numeric(raw_df[c], errors="coerce")

raw_df.to_csv(clean_path, index=False)
print(f"已生成清洗数据: {clean_path}")

## 第 3 部分：数据预览
字段覆盖：record_type / lot_id / 设备 / 班次 / 电流 / 电镀时长 / 温度 / 良率 / 缺陷标记 / 检验状态。

In [ ]:
df = pd.read_csv("../data/aami_yield_demo_clean.csv")
display(df.head(10))
print(df.dtypes)

## 第 4 部分：先看现象，不急着上 AI

In [ ]:
obs = df[df["inspection_status"] == "DONE"].copy()

# 图1：不同设备平均良率
machine_mean = obs.groupby("machine_id")["yield_pct"].mean().sort_values()
plt.figure(); plt.bar(machine_mean.index, machine_mean.values)
plt.title("不同设备平均良率")
plt.xlabel("machine_id"); plt.ylabel("avg yield_pct")
plt.show()

In [ ]:
# 图2：白班 vs 夜班
shift_mean = obs.groupby("shift")["yield_pct"].mean().reindex(["Day","Night"])
plt.figure(); plt.bar(shift_mean.index, shift_mean.values)
plt.title("白班 vs 夜班平均良率")
plt.xlabel("shift"); plt.ylabel("avg yield_pct")
plt.show()

In [ ]:
# 图3：电流 vs 良率
plt.figure(); plt.scatter(obs["plating_current_a"], obs["yield_pct"])
plt.title("电流 vs 良率")
plt.xlabel("plating_current_a"); plt.ylabel("yield_pct")
plt.show()

In [ ]:
# 图4：setup 后第几批 vs 良率
seq_mean = obs.groupby("setup_batch_seq")["yield_pct"].mean()
plt.figure(); plt.plot(seq_mean.index, seq_mean.values, marker="o")
plt.title("setup 后第几批 vs 平均良率")
plt.xlabel("setup_batch_seq"); plt.ylabel("avg yield_pct")
plt.show()

print("观察总结：E03、Night、高电流区间对应更低良率，且与已知异常批次特征一致。")

## 第 5 部分：定义 AI 任务
- 本 Demo 不做全厂优化。
- 任务聚焦：识别 **高风险 lot**，尤其是当前尚未终检的 PENDING lot。

## 第 6 部分：本地风险模型
使用 HIST 数据训练一个稳妥可复现的二分类模型，输出 risk_score 与 risk_level。

In [ ]:
train_df = df[df["record_type"] == "HIST"].copy()
train_df["target"] = (train_df["defect_flag"] == 1).astype(int)

feature_cols = ["machine_id", "shift", "setup_batch_seq", "plating_current_a", "plating_time_s", "bath_temp_c"]
cat_cols = ["machine_id", "shift"]
num_cols = [c for c in feature_cols if c not in cat_cols]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=200, class_weight="balanced", random_state=42)),
])

model.fit(train_df[feature_cols], train_df["target"])

current_pending = df[(df["record_type"] == "CURRENT") & (df["inspection_status"] == "PENDING")].copy()
current_pending["risk_score"] = model.predict_proba(current_pending[feature_cols])[:, 1]

# 阈值按演示可解释性设定：>=0.60 为高风险
current_pending["risk_level"] = np.where(current_pending["risk_score"] >= 0.60, "HIGH", "LOW")

display(current_pending[["lot_id", "machine_id", "shift", "plating_current_a", "risk_score", "risk_level"]].sort_values("risk_score", ascending=False))

assert current_pending.loc[current_pending["lot_id"] == "LOT1008", "risk_level"].iloc[0] == "HIGH", "LOT1008 需要被识别为高风险"
print("校验通过：LOT1008 已识别为高风险")

## 第 7 部分：LLM 解释层（支持开关）

In [ ]:
# ===== LLM 解释开关 =====
USE_LLM = True
LLM_PROVIDER = "auto"   # auto / openai / gemini / local

resolved_provider = get_llm_provider(provider=LLM_PROVIDER, use_llm=USE_LLM)
provider_cn = {
    "openai": "OpenAI 生成",
    "gemini": "Gemini 生成",
    "local": "本地模板生成",
}.get(resolved_provider, "本地模板生成")
print(f"当前解释方式：{provider_cn}")

In [ ]:
lot1008 = current_pending[current_pending["lot_id"] == "LOT1008"].iloc[0].to_dict()

similar_cases = (
    df[(df["record_type"] == "HIST") & (df["lot_id"].isin(["LOT0921", "LOT0934", "LOT0945"]))]
    [["lot_id", "machine_id", "shift", "plating_current_a", "yield_pct", "defect_type"]]
    .to_dict(orient="records")
)

explanation = generate_lot_explanation(
    lot_features=lot1008,
    similar_cases=similar_cases,
    provider=LLM_PROVIDER,
    use_llm=USE_LLM,
)

print("LOT1008 风险解释（结构化）")
print(json.dumps(explanation, ensure_ascii=False, indent=2))

## 第 8 部分：建议动作（建议，不自动执行）

In [ ]:
actions = generate_action_advice(
    lot_features=lot1008,
    provider=LLM_PROVIDER,
    use_llm=USE_LLM,
)

print("LOT1008 建议动作（结构化）")
print(json.dumps(actions, ensure_ascii=False, indent=2))

print("
说明：以上为建议动作，不自动执行，需要工程师确认。")

## 第 9 部分：闭环总结
**最小闭环：**
历史数据 -> 风险识别 -> 原因解释 -> 建议动作 -> 工程师确认 -> 结果回写 -> 持续学习

**产品化总结：**
> AI 不只是报表，它开始具备像工程师一样看 lot 风险的能力。